# SGD Regressor

In [152]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import eda
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor, Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.exceptions import ConvergenceWarning
import warnings

## Full Dataset

In [153]:
sbd = pd.read_csv("SeoulBikeData.csv", encoding = "latin-1")
sbd = eda.transform_data(sbd)

In [155]:
X_train, X_test, Y_train, Y_test = eda.split(sbd)

In [156]:
sgd_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("sgd", SGDRegressor(max_iter = 2000, random_state = 42))
])

sgd_param_grid ={
    "sgd__alpha": [0.0001, 0.001, 0.01, 0.1],
    "sgd__learning_rate": ["invscaling", "adaptive"],
    "sgd__penalty": ["l2", "l1", "elasticnet"]
}

sgd_cv = GridSearchCV(
    sgd_pipeline, sgd_param_grid,
    cv = TimeSeriesSplit(n_splits = 3),
    scoring = "neg_root_mean_squared_error",
    n_jobs = -1
)

sgd_cv.fit(X_train, Y_train)
print(f"Best Parameters: {sgd_cv.best_params_}")
print()

sgd_preds = np.clip(sgd_cv.best_estimator_.predict(X_test), 0, None)
sgd_rmse = np.sqrt(mean_squared_error(sgd_preds, Y_test))
sgd_mae = mean_absolute_error(sgd_preds, Y_test)
sgd_r2 = r2_score(sgd_preds, Y_test)
print("SGD Best Parameters Metrics")
print(f"RMSE: {sgd_rmse:.4f}")
print(f"MAE: {sgd_mae:.4f}")
print(f"R^2: {sgd_r2:.4f}")


Best Parameters: {'sgd__alpha': 0.1, 'sgd__learning_rate': 'invscaling', 'sgd__penalty': 'l1'}

SGD Best Parameters Metrics
RMSE: 0.4161
MAE: 0.3006
R^2: 0.7568


In [157]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

ridge_param_grid = {"ridge__alpha": [0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0, 500.0]}

ridge_cv = GridSearchCV(
    ridge_pipeline, ridge_param_grid,
    cv = TimeSeriesSplit(n_splits = 3),
    scoring = "neg_root_mean_squared_error",
    n_jobs = -1
)

ridge_cv.fit(X_train, Y_train)
print(f"Best Parameters: {ridge_cv.best_params_}")
print()

ridge_preds = np.clip(ridge_cv.best_estimator_.predict(X_test), 0, None)
ridge_rmse = np.sqrt(mean_squared_error(ridge_preds, Y_test))
ridge_mae = mean_absolute_error(ridge_preds, Y_test)
ridge_r2 = r2_score(ridge_preds, Y_test)
print("Ridge Best Parameters Metrics")
print(f"RMSE: {ridge_rmse:.4f}")
print(f"MAE: {ridge_mae:.4f}")
print(f"R^2: {ridge_r2:.4f}")


Best Parameters: {'ridge__alpha': 5.0}

Ridge Best Parameters Metrics
RMSE: 0.3963
MAE: 0.2895
R^2: 0.8218
